In [47]:
import json
import io
path = "courses.json"
def load_json_with_fallback(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except UnicodeDecodeError:
        try:
            with open(path, "r", encoding="utf-8-sig") as f:
                return json.load(f)
        except Exception:
            with open(path, "rb") as f:
                raw = f.read()
            try:
                text = raw.decode("utf-8")
            except Exception:
                text = raw.decode("latin-1", errors="replace")
            return json.loads(text)

courses = load_json_with_fallback(path)

In [48]:
# preprocess prerequisites into a structured machine-readable tree
import re


class ReqNode:
    def __init__(self, value):
        self.value = value

    def __repr__(self) -> str:
        return f"ReqNode({self.value!r})"

    def __or__(self, other):
        if isinstance(other, ReqNode):
            return UnionNode([self, other])
        raise TypeError("Can only use | operator with another ReqNode")


class CourseNode(ReqNode):
    value: tuple[str, str]

    def __repr__(self) -> str:
        return f"CourseNode({self.value!r})"

    def __str__(self) -> str:
        return f"{self.value[0]} {self.value[1]}"


class UnionNode(ReqNode):
    value: list[ReqNode]

    def __repr__(self) -> str:
        return f"UnionNode({self.value!r})"


class AndNode(ReqNode):
    value: list[ReqNode]

    def __repr__(self) -> str:
        return f"AndNode({self.value!r})"


COURSE_EXACT_PATTERN = re.compile(r"^([A-Z]{2,4})\s*(\d{4})$")
COURSE_ANY_PATTERN = re.compile(r"\b[A-Z]{2,4}\s*\d{4}\b")
IMPLIED_DEPT_LIST_PATTERN = re.compile(
    r"^([A-Z]{2,4})\s*((?:\d{4}\s*,\s*)*\d{4}\s*,?\s*or\s*\d{4})$",
    re.IGNORECASE,
)
OP_PATTERN = re.compile(r"(?i)\b(and|or)\b")
PERMISSION_PATTERN = re.compile(
    r"(?i)\b(instructor permission|permission of instructor|consent of instructor|instructor consent)\b"
)


def _parse_implied_department_list(text: str) -> UnionNode | None:
    match = IMPLIED_DEPT_LIST_PATTERN.fullmatch(text)
    if not match:
        return None

    dept = match.group(1).upper()
    number_chunk = match.group(2)
    numbers = re.findall(r"\d{4}", number_chunk)
    if len(numbers) < 2:
        return None

    return UnionNode([CourseNode((dept, num)) for num in numbers])


def _to_node(text: str) -> ReqNode:
    text = text.strip(" ,.;")

    implied_dept_union = _parse_implied_department_list(text)
    if implied_dept_union is not None:
        return implied_dept_union

    match = COURSE_EXACT_PATTERN.fullmatch(text)
    if match:
        return CourseNode((match.group(1), match.group(2)))
    return ReqNode(text)


def _has_permission_phrase(fragment: str) -> bool:
    return PERMISSION_PATTERN.search(fragment) is not None


def _is_logical_operator(left_fragment: str, right_fragment: str, op: str) -> bool:
    left = left_fragment.strip()
    right = right_fragment.strip()
    if not left or not right:
        return False

    # Avoid splitting narrative lists like "..., CS 1112, or 1113".
    if left.endswith(",") or right.startswith(","):
        return False

    left_has_course = COURSE_ANY_PATTERN.search(left) is not None
    right_has_course = COURSE_ANY_PATTERN.search(right) is not None
    left_group = left.endswith(")")
    right_group = right.startswith("(")
    left_has_permission = _has_permission_phrase(left)
    right_has_permission = _has_permission_phrase(right)

    if op == "and":
        return (left_has_course or left_group) and (right_has_course or right_group)

    return (
        (left_has_course and right_has_course)
        or (left_group and right_group)
        or ((left_has_course or left_group) and right_has_permission)
        or ((right_has_course or right_group) and left_has_permission)
    )


def _peek_right_fragment(expr: str, start: int) -> str:
    i = start
    n = len(expr)
    while i < n and expr[i].isspace():
        i += 1
    if i < n and expr[i] == "(":
        return "("
    j = i
    while j < n:
        if expr[j] in "()":
            break
        match = OP_PATTERN.match(expr, j)
        if match:
            break
        j += 1
    return expr[i:j]


def _tokenize(expr: str) -> list[tuple[str, str]]:
    tokens: list[tuple[str, str]] = []
    buffer: list[str] = []
    i = 0
    n = len(expr)

    def flush_text():
        text = "".join(buffer).strip()
        if text:
            tokens.append(("TEXT", text))
        buffer.clear()

    while i < n:
        ch = expr[i]

        if ch == "(":
            flush_text()
            tokens.append(("LPAREN", ch))
            i += 1
            continue

        if ch == ")":
            flush_text()
            tokens.append(("RPAREN", ch))
            i += 1
            continue

        op_match = OP_PATTERN.match(expr, i)
        if op_match:
            op_text = op_match.group(1)
            op = op_text.lower()
            left_fragment = "".join(buffer)
            if not left_fragment.strip() and tokens and tokens[-1][0] == "RPAREN":
                left_fragment = ")"

            right_fragment = _peek_right_fragment(expr, op_match.end())
            if not right_fragment.strip():
                j = op_match.end()
                while j < n and expr[j].isspace():
                    j += 1
                if j < n and expr[j] == "(":
                    right_fragment = "("

            if _is_logical_operator(left_fragment, right_fragment, op):
                flush_text()
                tokens.append((op.upper(), op))
            else:
                buffer.append(op_text)

            i = op_match.end()
            continue

        buffer.append(ch)
        i += 1

    flush_text()
    return tokens


def _collapse_union(nodes: list[ReqNode]) -> ReqNode:
    flat: list[ReqNode] = []
    for node in nodes:
        if isinstance(node, UnionNode):
            flat.extend(node.value)
        else:
            flat.append(node)
    if len(flat) == 1:
        return flat[0]
    return UnionNode(flat)


def _collapse_and(nodes: list[ReqNode]) -> ReqNode:
    flat: list[ReqNode] = []
    for node in nodes:
        if isinstance(node, AndNode):
            flat.extend(node.value)
        else:
            flat.append(node)
    if len(flat) == 1:
        return flat[0]
    return AndNode(flat)


class PrereqParser:
    def __init__(self, expr: str):
        self.tokens = _tokenize(expr)
        self.i = 0

    def _peek(self):
        if self.i >= len(self.tokens):
            return None
        return self.tokens[self.i]

    def _consume(self):
        token = self._peek()
        if token is None:
            return None
        self.i += 1
        return token

    def parse(self) -> ReqNode:
        if not self.tokens:
            return ReqNode("")
        node = self._parse_or()
        if self._peek() is not None:
            raise ValueError(f"Unexpected token near {self._peek()}")
        return node

    def _parse_or(self) -> ReqNode:
        left = self._parse_and()
        options = [left]
        while self._peek() and self._peek()[0] == "OR":
            self._consume()
            options.append(self._parse_and())
        return _collapse_union(options)

    def _parse_and(self) -> ReqNode:
        left = self._parse_atom()
        terms = [left]
        while self._peek() and self._peek()[0] == "AND":
            self._consume()
            terms.append(self._parse_atom())
        return _collapse_and(terms)

    def _parse_atom(self) -> ReqNode:
        token = self._peek()
        if token is None:
            raise ValueError("Unexpected end of expression")

        token_type, token_text = token
        if token_type == "LPAREN":
            self._consume()
            node = self._parse_or()
            close = self._consume()
            if close is None or close[0] != "RPAREN":
                raise ValueError("Missing closing parenthesis")
            return node

        if token_type == "TEXT":
            self._consume()
            return _to_node(token_text)

        raise ValueError(f"Unexpected token {token_text!r}")


def parse_prerequisite(expr: str) -> ReqNode | list:
    if not isinstance(expr, str):
        return []
    cleaned = " ".join(expr.split())
    if not cleaned or cleaned.lower().startswith("none"):
        return []

    parser = PrereqParser(cleaned)
    try:
        return parser.parse()
    except ValueError:
        # Keep unparseable text as a leaf requirement so preprocessing never drops data.
        return ReqNode(cleaned)


for course_code, course in courses.items():
    prereq_str = course.get("prerequisites", "")
    course["prerequisites_parsed"] = parse_prerequisite(prereq_str)

In [49]:
# examples from spec + problematic narrative prerequisite
examples = [
    "(STAT 1100 or STAT 1120 or STAT 2020 or STAT 2120 or STAT 3120 or APMA 3110 or APMA 3120) and (STAT 1601 or STAT 1602 or STAT 3250 or CS 1110 or CS 1111 or CS 1112 or CS 1113)",
    "(STAT 3220 or STAT 4120 or STAT 5120 or ECON 3720 or ECON 4720 or SYS 4021) and (STAT 1601 or STAT 1602 or STAT 3080 or STAT 3250 or CS 1110 or CS 1111 or CS 1112 or CS 1113)",
    "3rd or 4th year Psychology or Cognitive Science major",
    "(One semester of calculus) and (PHYS 1710 or PHYS 1420 or PHYS 1425 or PHYS 2010)",
    "Some prior programming experience in any language. Students may only receive credit for one of CS 1110, 1111, 1112, or 1113.",
    "MUSI 3390 or instructor permission",
    "MUSI 3390 or MUSI 4543 or MUSI 4547 or instructor permission",
    "CS 1110, 1111, 1112, or 1113.",
]

for idx, text in enumerate(examples, start=1):
    print(f"Example {idx}: {text}")
    print(parse_prerequisite(text))
    print("-" * 80)

Example 1: (STAT 1100 or STAT 1120 or STAT 2020 or STAT 2120 or STAT 3120 or APMA 3110 or APMA 3120) and (STAT 1601 or STAT 1602 or STAT 3250 or CS 1110 or CS 1111 or CS 1112 or CS 1113)
AndNode([UnionNode([CourseNode(('STAT', '1100')), CourseNode(('STAT', '1120')), CourseNode(('STAT', '2020')), CourseNode(('STAT', '2120')), CourseNode(('STAT', '3120')), CourseNode(('APMA', '3110')), CourseNode(('APMA', '3120'))]), UnionNode([CourseNode(('STAT', '1601')), CourseNode(('STAT', '1602')), CourseNode(('STAT', '3250')), CourseNode(('CS', '1110')), CourseNode(('CS', '1111')), CourseNode(('CS', '1112')), CourseNode(('CS', '1113'))])])
--------------------------------------------------------------------------------
Example 2: (STAT 3220 or STAT 4120 or STAT 5120 or ECON 3720 or ECON 4720 or SYS 4021) and (STAT 1601 or STAT 1602 or STAT 3080 or STAT 3250 or CS 1110 or CS 1111 or CS 1112 or CS 1113)
AndNode([UnionNode([CourseNode(('STAT', '3220')), CourseNode(('STAT', '4120')), CourseNode(('STAT'